In [ ]:
import sys
sys.path.append("../../")

import math

from lib.data.dataplot import *
from lib.dds.dds import *
from lib.utils.time import *
from lib.system.controllers import *


MAX_DRONES = 6
CURRENT_DRONES = 6
HOVER_ALTITUDE = 1.0
FORMATION_SPACING = 1.0
RUN_SECONDS = 15
MIN_DELTA_T = 0.0001

FORMATION_NONE = 0
FORMATION_LINE = 1
FORMATION_TRIANGLE = 2
FORMATION_SQUARE = 3
FORMATION_CIRCLE = 4

FORMATION_NAMES = {
    FORMATION_NONE: "NONE",
    FORMATION_LINE: "LINE",
    FORMATION_TRIANGLE: "TRIANGLE",
    FORMATION_SQUARE: "SQUARE",
    FORMATION_CIRCLE: "CIRCLE",
}

STATE_NAMES = ["X", "Y", "Z", "TX", "TY", "TZ", "VX", "VY", "VZ", "WX", "WY", "WZ"]
FORCE_NAMES = ["f1", "f2", "f3", "f4"]
FORMATION_ORIGIN = {"x": 0.0, "y": 0.0, "z": HOVER_ALTITUDE}
PRIMARY_EVIDENCE_DRONE_ID = 0


def validate_startup_settings():
    if MAX_DRONES <= 0:
        raise ValueError("MAX_DRONES must be positive")
    if CURRENT_DRONES < 0 or CURRENT_DRONES > MAX_DRONES:
        raise ValueError("CURRENT_DRONES must be between 0 and MAX_DRONES")


def topic(drone_id, name):
    return f"D{drone_id}_{name}"


def hover_target_for(drone_id, drone_count, spacing=FORMATION_SPACING):
    # NONE keeps the old stable hover slots instead of forcing a shape.
    centered_index = drone_id - ((drone_count - 1) * 0.5)
    return {
        "x": 0.0,
        "y": centered_index * spacing,
        "z": HOVER_ALTITUDE,
    }


def zero_offset():
    return {"x": 0.0, "y": 0.0, "z": 0.0}


def line_offset_for(drone_id, drone_count, spacing=FORMATION_SPACING):
    centered_index = drone_id - ((drone_count - 1) * 0.5)
    return {"x": centered_index * spacing, "y": 0.0, "z": 0.0}


def triangle_offset_for(drone_id, drone_count, spacing=FORMATION_SPACING):
    if drone_count <= 1:
        return zero_offset()

    row = 0
    first_id_in_row = 0
    while drone_id >= first_id_in_row + row + 1:
        first_id_in_row += row + 1
        row += 1

    drones_in_row = row + 1
    column = drone_id - first_id_in_row
    x = (column - ((drones_in_row - 1) * 0.5)) * spacing
    y = (row - 1) * spacing
    return {"x": x, "y": y, "z": 0.0}


def square_offset_for(drone_id, drone_count, spacing=FORMATION_SPACING):
    columns = math.ceil(math.sqrt(max(drone_count, 1)))
    rows = math.ceil(max(drone_count, 1) / columns)
    row = drone_id // columns
    column = drone_id % columns
    x = (column - ((columns - 1) * 0.5)) * spacing
    y = (row - ((rows - 1) * 0.5)) * spacing
    return {"x": x, "y": y, "z": 0.0}


def circle_offset_for(drone_id, drone_count, spacing=FORMATION_SPACING):
    if drone_count <= 1:
        return zero_offset()

    angle = 2.0 * math.pi * drone_id / drone_count
    radius = spacing
    return {"x": radius * math.cos(angle), "y": radius * math.sin(angle), "z": 0.0}


def formation_offset_for(drone_id, drone_count, formation_id, spacing=FORMATION_SPACING):
    if formation_id == FORMATION_LINE:
        return line_offset_for(drone_id, drone_count, spacing)
    if formation_id == FORMATION_TRIANGLE:
        return triangle_offset_for(drone_id, drone_count, spacing)
    if formation_id == FORMATION_SQUARE:
        return square_offset_for(drone_id, drone_count, spacing)
    if formation_id == FORMATION_CIRCLE:
        return circle_offset_for(drone_id, drone_count, spacing)
    return zero_offset()


def formation_target_for(drone_id, drone_count, formation_id, spacing=FORMATION_SPACING):
    if formation_id == FORMATION_NONE:
        return hover_target_for(drone_id, drone_count, spacing)

    offset = formation_offset_for(drone_id, drone_count, formation_id, spacing)
    return {
        "x": FORMATION_ORIGIN["x"] + offset["x"],
        "y": FORMATION_ORIGIN["y"] + offset["y"],
        "z": FORMATION_ORIGIN["z"] + offset["z"],
    }


def assert_target_close(actual, expected, tolerance=0.000001):
    for axis in ["x", "y", "z"]:
        if abs(actual[axis] - expected[axis]) > tolerance:
            raise AssertionError(f"{axis} was {actual[axis]}, expected {expected[axis]}")


def validate_formation_offsets():
    assert_target_close(formation_target_for(0, 4, FORMATION_LINE), {"x": -1.5, "y": 0.0, "z": HOVER_ALTITUDE})
    assert_target_close(formation_target_for(3, 4, FORMATION_LINE), {"x": 1.5, "y": 0.0, "z": HOVER_ALTITUDE})

    assert_target_close(formation_target_for(0, 3, FORMATION_TRIANGLE), {"x": 0.0, "y": -1.0, "z": HOVER_ALTITUDE})
    assert_target_close(formation_target_for(1, 3, FORMATION_TRIANGLE), {"x": -0.5, "y": 0.0, "z": HOVER_ALTITUDE})
    assert_target_close(formation_target_for(2, 3, FORMATION_TRIANGLE), {"x": 0.5, "y": 0.0, "z": HOVER_ALTITUDE})

    assert_target_close(formation_target_for(0, 4, FORMATION_SQUARE), {"x": -0.5, "y": -0.5, "z": HOVER_ALTITUDE})
    assert_target_close(formation_target_for(3, 4, FORMATION_SQUARE), {"x": 0.5, "y": 0.5, "z": HOVER_ALTITUDE})

    assert_target_close(formation_target_for(0, 4, FORMATION_CIRCLE), {"x": 1.0, "y": 0.0, "z": HOVER_ALTITUDE})
    assert_target_close(formation_target_for(1, 4, FORMATION_CIRCLE), {"x": 0.0, "y": 1.0, "z": HOVER_ALTITUDE})


def safe_value(value, default=0.0):
    return default if value is None else value


def distance_to_target(state, target):
    dx = target["x"] - state["x"]
    dy = target["y"] - state["y"]
    dz = target["z"] - state["z"]
    return math.sqrt((dx * dx) + (dy * dy) + (dz * dz))


def minimum_inter_drone_distance(states_by_drone_id):
    active_states = list(states_by_drone_id.values())
    if len(active_states) < 2:
        return None

    minimum_distance = None
    for first_index in range(len(active_states)):
        for second_index in range(first_index + 1, len(active_states)):
            distance = distance_to_target(active_states[first_index], active_states[second_index])
            if minimum_distance is None or distance < minimum_distance:
                minimum_distance = distance
    return minimum_distance


def format_formation_ids(formation_ids):
    return ", ".join(FORMATION_NAMES[formation_id] for formation_id in sorted(formation_ids))


class Multirotor:

    def __init__(self, x_target=0.0, y_target=0.0, z_target=HOVER_ALTITUDE):
        self.vz_control = PID_Controller(5.0, 10.0, 0.0, 5)
        self.z_control = PID_Controller(2.0, 0.0, 0.0, 2)

        self.w_roll_control = PID_Controller(0.75, 0.3, 0.0075, 2)
        self.roll_control = PID_Controller(1.0, 0.0, 0.0, 2)

        self.w_pitch_control = PID_Controller(0.75, 0.3, 0.0075, 2)
        self.pitch_control = PID_Controller(1.0, 0.0, 0.0, 2)

        self.vy_control = PID_Controller(0.4, 0.01, 0.25, math.radians(30))
        self.y_control = PID_Controller(1.0, 0.0, 0.0, 2.0)

        self.vx_control = PID_Controller(0.4, 0.01, 0.25, math.radians(30))
        self.x_control = PID_Controller(1.0, 0.0, 0.0, 2.0)

        self.z_target = z_target
        self.x_target = x_target
        self.y_target = y_target

        self.vz_target = 0.0
        self.vx_target = 0.0
        self.vy_target = 0.0

    def set_target(self, target):
        self.x_target = target["x"]
        self.y_target = target["y"]
        self.z_target = target["z"]

    def evaluate(self, delta_t, z, vz, x, vx, y, vy, roll, roll_rate, pitch, pitch_rate):
        # The output order matches the four propeller topics D{i}_f1..D{i}_f4.
        self.vz_target = self.z_control.evaluate(delta_t, self.z_target - z)
        f = self.vz_control.evaluate(delta_t, self.vz_target - vz)

        self.vy_target = self.y_control.evaluate(delta_t, self.y_target - y)
        self.roll_target = -self.vy_control.evaluate(delta_t, self.vy_target - vy)

        self.vx_target = self.x_control.evaluate(delta_t, self.x_target - x)
        self.pitch_target = self.vx_control.evaluate(delta_t, self.vx_target - vx)

        self.roll_rate_target = self.roll_control.evaluate(delta_t, self.roll_target - roll)
        roll_command = self.w_roll_control.evaluate(delta_t, self.roll_rate_target - roll_rate)

        self.pitch_rate_target = self.pitch_control.evaluate(delta_t, self.pitch_target - pitch)
        pitch_command = self.w_pitch_control.evaluate(delta_t, self.pitch_rate_target - pitch_rate)

        return (
            f + roll_command - pitch_command,
            f - roll_command - pitch_command,
            f - roll_command + pitch_command,
            f + roll_command + pitch_command,
        )


def subscribe_topics_for(max_drones):
    topics = ["start", "tick", "drone_count", "formation_id"]
    for drone_id in range(max_drones):
        topics.append(topic(drone_id, "active"))
        for state_name in STATE_NAMES:
            topics.append(topic(drone_id, state_name))
    return topics


def read_drone_count(dds):
    raw_count = int(safe_value(dds.read("drone_count"), CURRENT_DRONES))
    return max(0, min(raw_count, MAX_DRONES))


def read_formation_id(dds):
    raw_formation_id = int(safe_value(dds.read("formation_id"), FORMATION_NONE))
    return max(FORMATION_NONE, min(raw_formation_id, FORMATION_CIRCLE))


def is_drone_active(dds, drone_id):
    return int(safe_value(dds.read(topic(drone_id, "active")), 1)) == 1


def read_drone_state(dds, drone_id):
    return {
        "x": safe_value(dds.read(topic(drone_id, "X"))),
        "y": safe_value(dds.read(topic(drone_id, "Y"))),
        "z": safe_value(dds.read(topic(drone_id, "Z"))),
        "roll": safe_value(dds.read(topic(drone_id, "TX"))),
        "pitch": safe_value(dds.read(topic(drone_id, "TY"))),
        "vx": safe_value(dds.read(topic(drone_id, "VX"))),
        "vy": safe_value(dds.read(topic(drone_id, "VY"))),
        "vz": safe_value(dds.read(topic(drone_id, "VZ"))),
        "roll_rate": safe_value(dds.read(topic(drone_id, "WX"))),
        "pitch_rate": safe_value(dds.read(topic(drone_id, "WY"))),
    }


def publish_forces(dds, drone_id, forces):
    for force_name, force_value in zip(FORCE_NAMES, forces):
        dds.publish(topic(drone_id, force_name), force_value, DDS.DDS_TYPE_FLOAT)


def stop_all_motors(dds, max_drones):
    for drone_id in range(max_drones):
        publish_forces(dds, drone_id, (0.0, 0.0, 0.0, 0.0))


def create_drone_plotter(drone_id):
    plotter = DataPlotter()
    plotter.set_x("time (seconds)")
    plotter.add_y("target_z", f"D{drone_id} target_z")
    plotter.add_y("current_z", f"D{drone_id} current_z")
    plotter.add_y("distance", f"D{drone_id} distance_to_target")
    return plotter


def create_primary_x_plotter(drone_id):
    plotter = DataPlotter()
    plotter.set_x("time (seconds)")
    plotter.add_y("target_x", f"D{drone_id} target_x")
    plotter.add_y("current_x", f"D{drone_id} current_x")
    return plotter


def create_swarm_evidence_plotter():
    plotter = DataPlotter()
    plotter.set_x("time (seconds)")
    plotter.add_y("min_distance", "minimum inter-drone distance")
    plotter.add_y("formation_id", "selected formation id")
    plotter.add_y("drone_count", "active drone count")
    return plotter


def append_drone_plot(plotter, current_time, robot, state, target):
    plotter.append_x(current_time)
    plotter.append_y("target_z", robot.z_target)
    plotter.append_y("current_z", state["z"])
    plotter.append_y("distance", distance_to_target(state, target))


def append_primary_x_plot(plotter, current_time, state, target):
    plotter.append_x(current_time)
    plotter.append_y("target_x", target["x"])
    plotter.append_y("current_x", state["x"])


def append_swarm_evidence_plot(plotter, current_time, states_by_drone_id, formation_id, drone_count):
    min_distance = minimum_inter_drone_distance(states_by_drone_id)
    if min_distance is None:
        return None

    plotter.append_x(current_time)
    plotter.append_y("min_distance", min_distance)
    plotter.append_y("formation_id", formation_id)
    plotter.append_y("drone_count", drone_count)
    return min_distance


validate_startup_settings()
validate_formation_offsets()

controllers = {}
drone_plotters = {}
primary_x_plotter = create_primary_x_plotter(PRIMARY_EVIDENCE_DRONE_ID)
swarm_evidence_plotter = create_swarm_evidence_plotter()
active_ids_previous_tick = set()
skipped_zero_delta_ticks = 0
formation_id_previous_tick = None
formations_seen = set()
drone_counts_seen = set()
minimum_distance_seen = None

dds = DDS()
dds.start()
dds.subscribe(subscribe_topics_for(MAX_DRONES))

print("Waiting for Godot start")
dds.wait("start")
print("Started")
print(f"Notebook configured for {CURRENT_DRONES} current drones and {MAX_DRONES} maximum drones")

t = Time()
t.start()

try:
    while t.get() < RUN_SECONDS:
        dds.wait("tick")
        delta_t = t.elapsed()
        if delta_t <= MIN_DELTA_T:
            skipped_zero_delta_ticks += 1
            continue

        current_time = t.get()
        drone_count = read_drone_count(dds)
        formation_id = read_formation_id(dds)
        formations_seen.add(formation_id)
        drone_counts_seen.add(drone_count)
        if formation_id != formation_id_previous_tick:
            print(f"Formation: {FORMATION_NAMES[formation_id]}")
            formation_id_previous_tick = formation_id

        active_ids_this_tick = set()
        states_this_tick = {}

        for drone_id in range(drone_count):
            if not is_drone_active(dds, drone_id):
                publish_forces(dds, drone_id, (0.0, 0.0, 0.0, 0.0))
                continue

            active_ids_this_tick.add(drone_id)
            if drone_id not in controllers:
                controllers[drone_id] = Multirotor()
            if drone_id not in drone_plotters:
                drone_plotters[drone_id] = create_drone_plotter(drone_id)

            target = formation_target_for(drone_id, drone_count, formation_id)
            robot = controllers[drone_id]
            robot.set_target(target)

            state = read_drone_state(dds, drone_id)
            states_this_tick[drone_id] = state
            forces = robot.evaluate(
                delta_t,
                state["z"], state["vz"],
                state["x"], state["vx"],
                state["y"], state["vy"],
                state["roll"], state["roll_rate"],
                state["pitch"], state["pitch_rate"],
            )
            publish_forces(dds, drone_id, forces)
            append_drone_plot(drone_plotters[drone_id], current_time, robot, state, target)
            if drone_id == PRIMARY_EVIDENCE_DRONE_ID:
                append_primary_x_plot(primary_x_plotter, current_time, state, target)

        min_distance = append_swarm_evidence_plot(
            swarm_evidence_plotter,
            current_time,
            states_this_tick,
            formation_id,
            drone_count,
        )
        if min_distance is not None and (minimum_distance_seen is None or min_distance < minimum_distance_seen):
            minimum_distance_seen = min_distance

        #add/remove support: explicitly zero any drone that disappeared.
        for removed_drone_id in active_ids_previous_tick - active_ids_this_tick:
            publish_forces(dds, removed_drone_id, (0.0, 0.0, 0.0, 0.0))

        active_ids_previous_tick = active_ids_this_tick

finally:
    stop_all_motors(dds, MAX_DRONES)
    dds.stop()

for drone_id in sorted(drone_plotters):
    if len(drone_plotters[drone_id].x_data) > 0:
        drone_plotters[drone_id].plot()

if len(primary_x_plotter.x_data) > 0:
    primary_x_plotter.plot()
if len(swarm_evidence_plotter.x_data) > 0:
    swarm_evidence_plotter.plot()

if formations_seen:
    print(f"Formations seen: {format_formation_ids(formations_seen)}")
if drone_counts_seen:
    print(f"Drone counts seen: {sorted(drone_counts_seen)}")
if minimum_distance_seen is not None:
    print(f"Minimum inter-drone distance: {minimum_distance_seen:.3f}")
else:
    print("Minimum inter-drone distance: not available")
if skipped_zero_delta_ticks > 0:
    print(f"Skipped {skipped_zero_delta_ticks} zero-delta tick(s)")

print("Done")

Waiting for Godot start
Started
Notebook configured for 6 current drones and 6 maximum drones
Formation: TRIANGLE
